# Cross-cohort ML (step 2): Terra driver

Calls the `ml/scripts` steps in order (spec addendum 2026-11-23: no Snakemake on Terra). Each step is a standalone script with explicit input and output paths; no analysis logic lives here. `ml/STEPS.md` lists the same steps with their inputs and outputs; keep the two in sync.

Run from the repository root. Participant-level files stay under `results/` on Terra; only the files marked **aggregate** may leave it.

In [ ]:
import os
os.chdir(os.path.expanduser("~/amp_pd"))   # repository root
MOFA = os.path.expanduser("~/mofa")        # MOFA working folder holding data/
AS = "results/ml/analysis_set"
PRE = "results/ml/preprocessed"
EN = "results/ml/ppmi_elastic_net"

## Issue 03: analysis set and coverage report

In [ ]:
!python3 ml/scripts/query_ml_fields.py --cohort PPMI --output {AS}/ppmi_ml_fields.csv
!python3 ml/scripts/query_ml_fields.py --cohort PDBP --output {AS}/pdbp_ml_fields.csv
!python3 ml/scripts/analysis_set.py --cohort PPMI --mofa_metadata {MOFA}/data/metadata_m0_PPMI.csv --ml_fields {AS}/ppmi_ml_fields.csv --rnaseq_matrix {MOFA}/data/ppmi_rnaseq_adjusted.csv --output {AS}/ppmi_participants.csv
!python3 ml/scripts/analysis_set.py --cohort PDBP --mofa_metadata {MOFA}/data/metadata_m0_PDBP.csv --ml_fields {AS}/pdbp_ml_fields.csv --rnaseq_matrix {MOFA}/data/pdbp_rnaseq_adjusted.csv --output {AS}/pdbp_participants.csv
!python3 ml/scripts/coverage_report.py --participants {AS}/ppmi_participants.csv {AS}/pdbp_participants.csv --output_dir {AS}/coverage_report   # aggregate

## Issue 04: label-free preprocessing and the internal PPMI AUC

1. Normalize each cohort on its own: size factors and log1p, then plate correction with `~ sex` (no diagnosis). PPMI keeps the genes passing MOFA's low-count filter; PDBP keeps all genes.
2. Choose the 5,000 most variable genes in PPMI only, check that no PDBP participant was used, and report the overlap with MOFA's 5,000 genes (aggregate).
3. Apply that list unchanged to both cohorts and standardize within cohort and sex.
4. Train the tracer elastic net on PPMI; internal nested-CV AUC overall and per sex (aggregate).

PDBP is prepared in step 3 but **not scored**: no PDBP classifier score is made in this issue.

In [ ]:
!python3 ml/scripts/normalize_cohort.py --cohort PPMI --counts {MOFA}/data/ppmi_rnaseq.csv --participants {AS}/ppmi_participants.csv --output {PRE}/ppmi_expr.csv
!python3 ml/scripts/normalize_cohort.py --cohort PDBP --counts {MOFA}/data/pdbp/raw/pdbp_rnaseq.csv --participants {AS}/pdbp_participants.csv --output {PRE}/pdbp_expr.csv

In [ ]:
# MOFA's 5,000 genes are the rows of its PPMI RNA-seq input matrix
!python3 ml/scripts/select_genes.py --expr {PRE}/ppmi_expr.csv --participants {AS}/ppmi_participants.csv --pdbp_participants {AS}/pdbp_participants.csv --mofa_genes {MOFA}/data/ppmi_rnaseq_adjusted.csv --n_top 5000 --genes_out {PRE}/genes_top5000.txt --report {PRE}/gene_selection_report.json

In [ ]:
!python3 ml/scripts/prepare_features.py --expr {PRE}/ppmi_expr.csv --participants {AS}/ppmi_participants.csv --genes {PRE}/genes_top5000.txt --output {PRE}/ppmi_features_top5000.csv
!python3 ml/scripts/prepare_features.py --expr {PRE}/pdbp_expr.csv --participants {AS}/pdbp_participants.csv --genes {PRE}/genes_top5000.txt --output {PRE}/pdbp_features_top5000.csv

In [ ]:
# Nested CV: 21 grid points x 10 inner x 5 outer folds on ~1,300 x 5,000; --n_jobs -1 uses all cores
!python3 ml/scripts/train_ppmi.py --features {PRE}/ppmi_features_top5000.csv --participants {AS}/ppmi_participants.csv --model {EN}/frozen_model.json --auc {EN}/internal_auc.tsv --oof {EN}/ppmi_oof_scores.csv --n_jobs -1

In [ ]:
# Aggregate results to bring back: the gene-selection report and the internal AUC table
import json, pandas as pd
print(json.dumps(json.load(open(f"{PRE}/gene_selection_report.json")), indent=2))
pd.read_csv(f"{EN}/internal_auc.tsv", sep="\t")